# Consolidación

In [2]:
from pyspark.sql import SparkSession, functions as F, types as T, Window

spark = (SparkSession.builder.appName("consolidacion")
         .config("spark.driver.memory","2g").enableHiveSupport().getOrCreate())
spark.conf.set("spark.sql.legacy.timeParserPolicy","LEGACY")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite","LEGACY")
spark.conf.set("spark.sql.shuffle.partitions","64")

REFINED = "/Obligatorio/refined"
RAW_OF  = "/Obligatorio/landing/openflights"
RAW_MB  = "/Obligatorio/landing/musicbrainz"

def rd(t): return spark.read.parquet(f"{REFINED}/{t}")
def norm(c): return F.lower(F.trim(F.regexp_replace(F.col(c), r"\s+", " ")))
def ckey(name_col): return F.sha2(norm(name_col), 256)
def haversine(la1,lo1,la2,lo2):
    a = (F.sin(F.radians(la2-la1)/2)**2
         + F.cos(F.radians(la1))*F.cos(F.radians(la2))*F.sin(F.radians(lo2-lo1)/2)**2)
    return F.lit(6371.0)*2*F.asin(F.sqrt(a))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-06-24T22:10:34,808 WARN [Thread-4] org.apache.hadoop.util.NativeCodeLoader - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# ===== Reconstrucción de dim_country (desde crudos) + re-clavado de dim_festival =====

RAW_OF = "/Obligatorio/landing/openflights"
RAW_MB = "/Obligatorio/landing/musicbrainz"
RAW_WD = "/Obligatorio/landing/wikidata/festivals.csv"

es2en = {
 "rusia":"Russia","reino unido":"United Kingdom","suiza":"Switzerland","eslovaquia":"Slovakia",
 "alemania":"Germany","españa":"Spain","albania":"Albania","estados unidos":"United States",
 "estados unidos de américa":"United States","francia":"France","italia":"Italy","países bajos":"Netherlands",
 "holanda":"Netherlands","bélgica":"Belgium","austria":"Austria","portugal":"Portugal","irlanda":"Ireland",
 "noruega":"Norway","suecia":"Sweden","finlandia":"Finland","dinamarca":"Denmark","polonia":"Poland",
 "república checa":"Czechia","chequia":"Czechia","hungría":"Hungary","grecia":"Greece","rumanía":"Romania",
 "bulgaria":"Bulgaria","croacia":"Croatia","serbia":"Serbia","eslovenia":"Slovenia","ucrania":"Ukraine",
 "turquía":"Turkey","japón":"Japan","china":"China","corea del sur":"South Korea","india":"India",
 "australia":"Australia","nueva zelanda":"New Zealand","canadá":"Canada","méxico":"Mexico","brasil":"Brazil",
 "argentina":"Argentina","chile":"Chile","uruguay":"Uruguay","colombia":"Colombia","perú":"Peru",
 "ecuador":"Ecuador","venezuela":"Venezuela","paraguay":"Paraguay","bolivia":"Bolivia","sudáfrica":"South Africa",
 "israel":"Israel","islandia":"Iceland","estonia":"Estonia","letonia":"Latvia","lituania":"Lithuania",
 "luxemburgo":"Luxembourg","malta":"Malta","chipre":"Cyprus",
}
mexpr = F.create_map([F.lit(x) for kv in es2en.items() for x in kv])
def norm_e(c): return F.lower(F.trim(F.regexp_replace(c, r"\s+", " ")))
def cc(col):
    s = F.trim(F.col(col))
    return F.when(s.isin(["","\\N","-","N/A"]), None).otherwise(s)

# --- 1) Reconstruir dim_country desde las tres fuentes ---
of_ap = spark.read.option("header",True).csv(f"{RAW_OF}/airports.csv").select(cc("country").alias("name"))
of_al = spark.read.option("header",True).csv(f"{RAW_OF}/airlines.csv").select(cc("country").alias("name"))

ca = spark.read.option("header",True).csv(f"{RAW_MB}/mb_country_area.csv")   # area_id
ar = spark.read.option("header",True).csv(f"{RAW_MB}/mb_area.csv")           # id, mbid, name
mb_c = ca.join(ar, ca.area_id == ar.id, "inner").select(F.col("name"))

wd_c = (spark.read.option("header",True).option("encoding","UTF-8").csv(RAW_WD)
        .select(F.col("countryLabel").alias("name"))
        .filter(F.col("name").isNotNull() & (F.trim("name") != ""))
        .withColumn("name", F.coalesce(mexpr[norm_e(F.col("name"))], F.col("name"))))

paises = (of_ap.unionByName(of_al).unionByName(mb_c).unionByName(wd_c)
          .filter(F.col("name").isNotNull() & (F.trim("name") != "")))

dim_country = (paises
    .withColumn("country_id", F.sha2(norm_e(F.col("name")), 256))
    .groupBy("country_id").agg(F.first("name", ignorenulls=True).alias("country_name"))
    .withColumn("country_iso", F.lit(None).cast("string"))
    .withColumn("source", F.lit("consolidado"))
    .select("country_id","country_name","country_iso","source"))
dim_country.write.mode("overwrite").parquet(f"{REFINED}/dim_country")
print("dim_country reconstruida:", rd("dim_country").count())

# --- 2) Re-clavar dim_festival.country_id desde el Wikidata crudo ---
wd = (spark.read.option("header",True).option("encoding","UTF-8").csv(RAW_WD)
      .select(F.regexp_extract("festival", r"/entity/(Q[0-9]+)", 1).alias("festival_id"),
              F.col("countryLabel").alias("country"))
      .filter(F.col("country").isNotNull() & (F.trim("country") != ""))
      .dropDuplicates(["festival_id"])
      .withColumn("english", F.coalesce(mexpr[norm_e(F.col("country"))], F.col("country")))
      .withColumn("country_id_canon", F.sha2(norm_e(F.col("english")), 256))
      .select("festival_id","country_id_canon"))

cols = ["festival_id","wikidata_id","festival_name","city_id","country_id","latitude","longitude",
        "start_date","capacity","musicbrainz_series_id","setlist_fm_id"]
fest2 = (rd("dim_festival").drop("country_id")
         .join(wd, "festival_id", "left")
         .withColumnRenamed("country_id_canon", "country_id")
         .select(*cols))
fest2.write.mode("overwrite").parquet(f"{REFINED}/dim_festival_tmp")
spark.read.parquet(f"{REFINED}/dim_festival_tmp").write.mode("overwrite").parquet(f"{REFINED}/dim_festival")
print("dim_festival re-clavada:", rd("dim_festival").count())




dim_country reconstruida: 370


dim_festival re-clavada: 992


In [5]:
# OpenFlights: ciudades con coords del aeropuerto
of_air = (spark.read.option("header",True)
    .schema("airport_id int,name string,city string,country string,iata string,icao string,"
            "latitude double,longitude double,altitude int,timezone double,dst string,"
            "tz_database string,type string,source string")
    .csv(f"{RAW_OF}/airports.csv")
    .filter((F.col("type")=="airport") & F.col("city").isNotNull() & F.col("country").isNotNull())
    .select(F.sha2(F.concat_ws("|", norm("country"), norm("city")),256).alias("city_id"),
            F.col("city").alias("city_name"), ckey("country").alias("country_id"),
            "latitude","longitude"))

# MusicBrainz: coords promedio por city_id, reusando la fórmula de city_id (país|ciudad)
area = spark.read.option("header",True).schema("id int,mbid string,name string").csv(f"{RAW_MB}/mb_area.csv")
place = (spark.read.option("header",True)
    .schema("id int,mbid string,name string,type int,area int,coordinates string").csv(f"{RAW_MB}/mb_place.csv")
    .withColumn("lat", F.regexp_extract("coordinates", r"\(([-0-9.]+),([-0-9.]+)\)",1).cast("double"))
    .withColumn("lon", F.regexp_extract("coordinates", r"\(([-0-9.]+),([-0-9.]+)\)",2).cast("double"))
    .filter(F.col("lat").between(-90,90) & F.col("lon").between(-180,180)))
mb_coords = (place.join(area.select(F.col("id").alias("area"), F.col("name").alias("city_name")),"area","left")
    .join(rd("dim_city").select("city_id","city_name"), "city_name", "inner")
    .groupBy("city_id").agg(F.avg("lat").alias("latitude"), F.avg("lon").alias("longitude")))

mb_city = rd("dim_city").drop("latitude","longitude").join(mb_coords, "city_id", "left")

dim_city = (of_air.unionByName(mb_city.select("city_id","city_name","country_id","latitude","longitude"))
    .groupBy("city_id")
    .agg(F.first("city_name", ignorenulls=True).alias("city_name"),
         F.first("country_id", ignorenulls=True).alias("country_id"),
         F.avg("latitude").alias("latitude"), F.avg("longitude").alias("longitude")))
dim_city.write.mode("overwrite").parquet(f"{REFINED}/dim_city_tmp")
spark.read.parquet(f"{REFINED}/dim_city_tmp").write.mode("overwrite").parquet(f"{REFINED}/dim_city")
print("dim_city consolidada:", rd("dim_city").count(),
      "| con coords:", rd("dim_city").filter(F.col("latitude").isNotNull()).count())

fest = rd("dim_festival").select("festival_id", F.col("latitude").alias("f_lat"), F.col("longitude").alias("f_lon")) \
        .filter(F.col("f_lat").isNotNull())
airp = rd("dim_airport").select("airport_id", F.col("latitude").alias("a_lat"), F.col("longitude").alias("a_lon")) \
        .filter(F.col("a_lat").isNotNull())
conn = rd("fact_airport_connectivity")

# Cross join (festivales pequeño -> broadcast). Nos quedamos con los 3 aeropuertos más cercanos.
pares = (F.broadcast(fest).crossJoin(airp)
    .withColumn("distance_km", haversine(F.col("f_lat"),F.col("f_lon"),F.col("a_lat"),F.col("a_lon"))))
w = Window.partitionBy("festival_id").orderBy("distance_km")
cercanos = pares.withColumn("airport_rank", F.row_number().over(w)).filter(F.col("airport_rank")<=3)

fact_festival_air_accessibility = (
    cercanos.join(conn, "airport_id", "left")
    .withColumn("direct_routes", F.coalesce(F.col("direct_routes_out"),F.lit(0))+F.coalesce(F.col("direct_routes_in"),F.lit(0)))
    .withColumn("accessibility_score",
        (F.coalesce(F.col("connectivity_score"),F.lit(0.0)) / (F.lit(1.0)+F.col("distance_km")/F.lit(50.0))))
    .select("festival_id","airport_id",
            F.round("distance_km",2).alias("distance_km"),"airport_rank",
            "connectivity_score","direct_routes","direct_countries",
            F.col("airlines_count"),"accessibility_score"))
fact_festival_air_accessibility.write.mode("overwrite").parquet(f"{REFINED}/fact_festival_air_accessibility")
print("fact_festival_air_accessibility:", rd("fact_festival_air_accessibility").count())



dim_city consolidada: 17244 | con coords: 13707


fact_festival_air_accessibility: 2976


In [4]:
fest = rd("dim_festival").select("festival_id", F.col("latitude").alias("f_lat"), F.col("longitude").alias("f_lon")) \
        .filter(F.col("f_lat").isNotNull())
cities = rd("dim_city").select("city_id", F.col("latitude").alias("c_lat"), F.col("longitude").alias("c_lon")) \
        .filter(F.col("c_lat").isNotNull())

w = Window.partitionBy("festival_id").orderBy("dist")
fest_city = (F.broadcast(fest).crossJoin(cities)
    .withColumn("dist", haversine(F.col("f_lat"),F.col("f_lon"),F.col("c_lat"),F.col("c_lon")))
    .withColumn("rn", F.row_number().over(w)).filter(F.col("rn")==1)
    .select("festival_id","city_id"))

fact_city_genre_activity = (
    fest_city.join(rd("bridge_festival_genre"), "festival_id")
    .groupBy("city_id","genre_id")
    .agg(F.countDistinct("festival_id").alias("festivals_count"))
    .withColumn("events_count", F.lit(0))     # los eventos no traen género en MusicBrainz
    .withColumn("artists_count", F.lit(0))    # idem
    .select("city_id","genre_id","events_count","artists_count","festivals_count"))
fact_city_genre_activity.write.mode("overwrite").parquet(f"{REFINED}/fact_city_genre_activity")
print("fact_city_genre_activity:", rd("fact_city_genre_activity").count())



fact_city_genre_activity: 305


In [5]:
SUDAMERICA = ["Argentina","Bolivia","Brazil","Chile","Colombia","Ecuador","Guyana",
              "Paraguay","Peru","Suriname","Uruguay","Venezuela"]

# Eventos en Sudamérica en los últimos 5 años (>=2021), con país del evento y artista participante.
ae = (rd("fact_artist_event")
    .withColumn("year", (F.col("event_date_id")/F.lit(10000)).cast("int"))
    .filter(F.col("year") >= 2021)
    .join(rd("dim_country").select(F.col("country_id").alias("event_country_id"),
                                   F.col("country_name").alias("event_country")), "event_country_id", "left")
    .filter(F.col("event_country").isin(SUDAMERICA)))

art = rd("dim_artist").select("artist_id","artist_type","origin_country_id",
                              "begin_year","end_year")

fact_south_america_tours = (
    ae.groupBy("artist_id", F.col("year").alias("tour_year"))
    .agg(
        F.countDistinct("event_country").alias("countries_visited_count"),
        F.max(F.when(F.col("event_country")=="Uruguay",True).otherwise(False)).alias("visited_uruguay"),
        F.max(F.when(F.col("event_country")=="Argentina",True).otherwise(False)).alias("visited_argentina"),
        F.max(F.when(F.col("event_country")=="Brazil",True).otherwise(False)).alias("visited_brazil"),
        F.max(F.when(F.col("event_country")=="Chile",True).otherwise(False)).alias("visited_chile"),
        F.max(F.when(F.col("event_country")=="Paraguay",True).otherwise(False)).alias("visited_paraguay"),
        F.countDistinct("event_id").alias("events_in_south_america"))
    .join(art, "artist_id", "left")
    .withColumn("artist_active_years", F.coalesce(F.col("end_year"),F.lit(2025)) - F.col("begin_year"))
    .select("artist_id","tour_year","countries_visited_count","visited_uruguay","visited_argentina",
            "visited_brazil","visited_chile","visited_paraguay","events_in_south_america",
            F.col("origin_country_id").alias("artist_origin_country_id"),"artist_type","artist_active_years"))
fact_south_america_tours.write.mode("overwrite").parquet(f"{REFINED}/fact_south_america_tours")
print("fact_south_america_tours:", rd("fact_south_america_tours").count())
print("Artistas que incluyeron Uruguay:",
      rd("fact_south_america_tours").filter(F.col("visited_uruguay")).select("artist_id").distinct().count())



fact_south_america_tours: 521
Artistas que incluyeron Uruguay: 29


In [8]:
tablas = ["dim_country","dim_city","dim_date","dim_airport","dim_airline","dim_festival","dim_genre",
          "dim_artist","dim_event","bridge_festival_genre","bridge_artist_event",
          "fact_air_route","fact_airport_connectivity","fact_festival_air_accessibility",
          "fact_airport_daily_ops","fact_airport_weekly_ops","fact_airport_yearly_baseline",
          "fact_artist_event","fact_country_music_flow","fact_city_genre_activity","fact_south_america_tours"]
for t in tablas:
    try:
        print(f"{t:34} filas={rd(t).count():>9}")
    except Exception as e:
        print(f"{t:34} FALTA / error")



dim_country                        filas=      370
dim_city                           filas=    17244
dim_date                           filas=    31411
dim_airport                        filas=     8264
dim_airline                        filas=     6162
dim_festival                       filas=      992
dim_genre                          filas=      102
dim_artist                         filas=   567202
dim_event                          filas=   117931
bridge_festival_genre              filas=      312
bridge_artist_event                filas=   234845
fact_air_route                     filas=    66707
fact_airport_connectivity          filas=     8264
fact_festival_air_accessibility    filas=     2976


[Stage 167:======================================>                  (4 + 2) / 6]

fact_airport_daily_ops             filas=   359248


fact_airport_weekly_ops            filas=    52352
fact_airport_yearly_baseline       filas=     1047
fact_artist_event                  filas=   234845
fact_country_music_flow            filas=    13220
fact_city_genre_activity           filas=      305
fact_south_america_tours           filas=      521


In [6]:
# ===== Housekeeping: mover crudos de landing -> raw (libera landing) =====
sc = spark.sparkContext
hadoop = sc._jvm.org.apache.hadoop.fs
fs = hadoop.FileSystem.get(sc._jsc.hadoopConfiguration())
Path = hadoop.Path

LANDING = "/Obligatorio/landing"
RAW = "/Obligatorio/raw"
fs.mkdirs(Path(RAW))

for fuente in ["openflights", "wikidata", "musicbrainz", "bts"]:
    src = Path(f"{LANDING}/{fuente}")
    dst = Path(f"{RAW}/{fuente}")
    if not fs.exists(src):
        print("ya no está en landing:", fuente); continue
    if fs.exists(dst):
        fs.delete(dst, True)          # limpia destino si ya existía
    fs.rename(src, dst)               # MUEVE landing -> raw
    print("movido a raw:", fuente)

print("\nContenido de raw:")
for s in fs.listStatus(Path(RAW)):
    print("   ", s.getPath().getName())

print("\nContenido de landing (debería quedar vacío):")
estado = fs.listStatus(Path(LANDING))
if len(estado) == 0:
    print("   (vacío)")
else:
    for s in estado:
        print("   ", s.getPath().getName())



movido a raw: openflights
movido a raw: wikidata
movido a raw: musicbrainz
movido a raw: bts

Contenido de raw:
    bts
    musicbrainz
    openflights
    wikidata

Contenido de landing (debería quedar vacío):
   (vacío)


In [7]:
from pyspark.sql import functions as F
REFINED = "/Obligatorio/refined"
def rd(t): return spark.read.parquet(f"{REFINED}/{t}")

tablas = ["dim_country","dim_city","dim_date","dim_airport","dim_airline","dim_festival","dim_genre",
          "dim_artist","dim_event","bridge_festival_genre","bridge_artist_event",
          "fact_air_route","fact_airport_connectivity","fact_festival_air_accessibility",
          "fact_airport_daily_ops","fact_airport_weekly_ops","fact_airport_yearly_baseline",
          "fact_artist_event","fact_country_music_flow","fact_city_genre_activity","fact_south_america_tours"]
print("== CONTEOS ==")
for t in tablas:
    try: print(f"  {t:32} {rd(t).count():>9}")
    except Exception: print(f"  {t:32}   FALTA/ERROR")

print("\n== COBERTURA Y CRUCES ==")
fest = rd("dim_festival"); ev = rd("dim_event")
dc_ids = rd("dim_country").select("country_id")
ap_ids = rd("dim_airport").select("airport_id")

# Festivales con aeropuerto cercano (Q1)
print(f"  Festivales con aeropuerto cercano: "
      f"{rd('fact_festival_air_accessibility').select('festival_id').distinct().count()} de {fest.count()}")

# País de festivales / eventos resuelto y válido en dim_country
print(f"  Festivales con país: {fest.filter(F.col('country_id').isNotNull()).count()} de {fest.count()} "
      f"| válidos en dim_country: {fest.filter(F.col('country_id').isNotNull()).join(dc_ids,'country_id','left_semi').count()}")
print(f"  Eventos con país:    {ev.filter(F.col('country_id').isNotNull()).count()} de {ev.count()} "
      f"| válidos en dim_country: {ev.filter(F.col('country_id').isNotNull()).join(dc_ids,'country_id','left_semi').count()}")
print(f"  Eventos con ciudad:  {ev.filter(F.col('city_id').isNotNull()).count()} de {ev.count()}")

# Artistas que incluyeron Uruguay (Q5)
sat = rd("fact_south_america_tours")
print(f"  Artistas con giras en Sudamérica: {sat.select('artist_id').distinct().count()} "
      f"| que incluyeron Uruguay: {sat.filter(F.col('visited_uruguay')).select('artist_id').distinct().count()}")

# Integridad: aeropuertos de BTS contra dim_airport (Q3)
print(f"  daily_ops con airport_id sin match en dim_airport: "
      f"{rd('fact_airport_daily_ops').select('airport_id').distinct().join(ap_ids,'airport_id','left_anti').count()}")

# Integridad de puentes
bae = rd("bridge_artist_event")
print(f"  bridge_artist_event huérfanos -> artista: "
      f"{bae.join(rd('dim_artist').select('artist_id'),'artist_id','left_anti').count()} | evento: "
      f"{bae.join(ev.select('event_id'),'event_id','left_anti').count()}")
bfg = rd("bridge_festival_genre")
print(f"  bridge_festival_genre huérfanos -> género: "
      f"{bfg.join(rd('dim_genre').select('genre_id'),'genre_id','left_anti').count()} | festival: "
      f"{bfg.join(fest.select('festival_id'),'festival_id','left_anti').count()}")

# Vistazo Q2: top flujos exportador -> receptor
print("\n  Top flujos país origen -> país evento (Q2):")
(rd("fact_country_music_flow")
 .join(rd("dim_country").select(F.col("country_id").alias("artist_origin_country_id"), F.col("country_name").alias("origen")), "artist_origin_country_id","left")
 .join(rd("dim_country").select(F.col("country_id").alias("event_country_id"),         F.col("country_name").alias("receptor")), "event_country_id","left")
 .groupBy("origen","receptor").agg(F.sum("events_count").alias("eventos"))
 .filter(F.col("origen") != F.col("receptor"))
 .orderBy(F.desc("eventos")).show(10, truncate=False))



== CONTEOS ==
  dim_country                            370
  dim_city                             17244
  dim_date                             31411
  dim_airport                           8264
  dim_airline                           6162
  dim_festival                           992
  dim_genre                              102
  dim_artist                          567202
  dim_event                           117931
  bridge_festival_genre                  312
  bridge_artist_event                 234845
  fact_air_route                       66707
  fact_airport_connectivity             8264
  fact_festival_air_accessibility       2976


  fact_airport_daily_ops              359248
  fact_airport_weekly_ops              52352
  fact_airport_yearly_baseline          1047
  fact_artist_event                   234845
  fact_country_music_flow              13220
  fact_city_genre_activity               305
  fact_south_america_tours               521

== COBERTURA Y CRUCES ==
  Festivales con aeropuerto cercano: 992 de 992
  Festivales con país: 927 de 992 | válidos en dim_country: 927
  Eventos con país:    102079 de 117931 | válidos en dim_country: 102079
  Eventos con ciudad:  102079 de 117931
  Artistas con giras en Sudamérica: 440 | que incluyeron Uruguay: 29


[Stage 206:======================================>                  (4 + 2) / 6]

  daily_ops con airport_id sin match en dim_airport: 0


  bridge_artist_event huérfanos -> artista: 0 | evento: 0
  bridge_festival_genre huérfanos -> género: 0 | festival: 0

  Top flujos país origen -> país evento (Q2):
+--------------+--------------+-------+
|origen        |receptor      |eventos|
+--------------+--------------+-------+
|United Kingdom|United States |4021   |
|United States |United Kingdom|3121   |
|United States |Canada        |1977   |
|United States |Germany       |1948   |
|United Kingdom|Germany       |1834   |
|Canada        |United States |1544   |
|United States |Netherlands   |1116   |
|United States |Belgium       |855    |
|United States |France        |736    |
|United Kingdom|Belgium       |703    |
+--------------+--------------+-------+
only showing top 10 rows

